In [0]:
%pip install kafka-python

In [0]:
dbutils.library.restartPython()

In [0]:
eh_namespace = "shruthi-eh-namespace.servicebus.windows.net:9093"
eventhub_name = "fraud-transactions"


connection_string = dbutils.secrets.get(scope="my-scope", key="eventhub-conn")

kafka_options = {
  "kafka.bootstrap.servers": eh_namespace,
  "subscribe": eventhub_name,

  "kafka.sasl.mechanism": "PLAIN",
  "kafka.security.protocol": "SASL_SSL",
  "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";',

  "startingOffsets": "earliest"
}

In [0]:
df = spark.readStream \
  .format("kafka") \
  .options(**kafka_options) \
  .load()

In [0]:
df_parsed = df.selectExpr("CAST(value AS STRING)")

In [0]:
from pyspark.sql.functions import from_json, col

schema = """
transaction_id INT,
amount INT,
is_fraud INT,
city STRING
"""

df_final = df_parsed.select(from_json(col("value"), schema).alias("data")).select("data.*")

In [0]:
df_final.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", "/Volumes/shruthi_databricks/default/fraud_data/checkpoints/fraud_stream") \
  .start("/Volumes/shruthi_databricks/default/fraud_data/bronze/fraud_stream")

In [0]:
df_test = spark.read \
  .format("kafka") \
  .option('kafka.bootstrap.servers', 'your.kafka.server:9092').options(**kafka_options) \
  .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";') \
  .load()

df_test.show()